# Reading and writing files

This module covers Python's file handling: paths, opening files, reading and writing text and bytes, and the patterns that come up when a tm1py script reads its config from disk, exports cube data to a CSV, appends to an audit log, or hands a blob upload to TM1. Readers are assumed to be comfortable with Python syntax and at least passing familiarity with TM1 cubes and the tm1py library; the focus here is on the file ecosystem around a script, not on what tm1py itself does over the wire.

A typical TM1 job touches more files than it touches REST endpoints. Before the script connects, it loads a config. While it runs, it appends to a log and may stream a CSV export to disk. After it finishes, it leaves behind exports, audit records, and sometimes a blob it uploaded along the way. Most of what a TM1 user needs from Python file I/O is the ability to open the right path with the right encoding, read or write the contents, and close the file without leaking handles.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. Where files show up around a tm1py script
2. Paths with `pathlib`
3. Opening a file with `open` and `with`
4. Text mode, binary mode, and encoding
5. Reading a whole file at once
6. Reading a file line by line
7. Writing and overwriting
8. Appending to a log file
9. CSV files
10. Listing a directory and globbing
11. Atomic writes
12. The working directory and resolving relative paths
13. Real world design principles
14. Common mistakes

---

## 1. Where files show up around a tm1py script

A tm1py script is rarely alone on disk. The directory around it usually contains a connection config, an MDX query or two saved as plain text, an output folder for exports, a log file that grows on every run, and sometimes test fixtures captured from a real TM1 server. Knowing which file lives where, and which encoding it uses, is half the battle.

```
sales_plan_job/
  run.py                    # the tm1py script
  config.json               # server address, view names, output dir
  mdx/
    sales_by_region.mdx     # MDX text used to query the cube
  exports/
    sales_plan_2026Q1.csv   # cube slice written out by the script
  blobs/
    forecast_upload.csv     # raw CSV the script uploads to TM1 as a blob
  logs/
    audit.log               # one line per run, append only
```

Three properties matter for everything that follows. First, files are sequences of bytes; text is a layer on top, decoded with an encoding. Second, every file handle must be closed; leaving one open is a resource leak that on Windows also locks the file against rename or delete. Third, the script's idea of "where am I" is the working directory of the process that started it, which is not always the directory the script lives in. The next sections take these one at a time.

## 2. Paths with `pathlib`

`pathlib.Path` is the modern way to handle filesystem paths in Python. A `Path` is an object that knows how to join with other paths, ask whether it exists, list the directory it points to, and read or write its own contents. It replaces a grab bag of older functions in `os` and `os.path` and works on Windows and POSIX without changes to the calling code.

In [ ]:
from pathlib import Path

job_dir = Path("/var/lib/tm1py/sales_plan_job")
config_file = job_dir / "config.json"
export_dir = job_dir / "exports"

config_file.exists()           # True
config_file.is_file()          # True
config_file.suffix             # '.json'
config_file.stem               # 'config'
config_file.parent             # PosixPath('/var/lib/tm1py/sales_plan_job')

export_dir.mkdir(exist_ok=True)

The `/` operator joins path segments, picking the right separator for the platform. `Path` instances are immutable, so `job_dir / "config.json"` returns a new `Path` rather than modifying `job_dir`. This makes them safe to pass around and to use as dictionary keys.

String paths still work with `open()` and most of the standard library, but mixing strings and `Path` objects across a codebase invites bugs around trailing slashes and platform separators. Pick `Path` and stay there.

## 3. Opening a file with `open` and `with`

`open()` returns a file object. The file object holds an operating system handle and a buffer. The handle must be released, and the buffer must be flushed, when the work is done. The `with` statement guarantees both, even if the block raises:

In [ ]:
from pathlib import Path

config_path = Path("config.json")

with config_path.open("r", encoding="utf-8") as f:
    text = f.read()

# f is closed here, even if read() raised

`Path.open` and the builtin `open(path, ...)` are equivalent; `Path.open` reads more cleanly when the path is already a `Path`. The first positional argument after the path is the mode string. The common modes are `"r"` for read, `"w"` for write (truncate first), `"a"` for append, `"x"` for write but fail if the file exists, and any of these with a `"b"` suffix for binary. Text modes accept an `encoding` keyword; binary modes do not.

Without `with`, the correct pattern is `try` / `finally`:

In [ ]:
f = config_path.open("r", encoding="utf-8")
try:
    text = f.read()
finally:
    f.close()

This is what `with` expands to. The context manager module covers the mechanism in detail; for files, the takeaway is that every `open` should be paired with a `with`. A bare `open()` whose result is assigned to a variable and never closed will eventually leak, and on Windows will block other processes from touching the file until the Python process exits.

## 4. Text mode, binary mode, and encoding

A file on disk is bytes. Text mode wraps the file in a decoder so that reads return `str` and writes accept `str`. Binary mode skips that layer and works in `bytes`. The mode is chosen at open time and cannot be mixed:

In [ ]:
with Path("sales_plan_2026Q1.csv").open("r", encoding="utf-8") as f:
    header = f.readline()
    type(header)   # <class 'str'>

with Path("forecast_upload.csv").open("rb") as f:
    raw = f.read()
    type(raw)      # <class 'bytes'>

The `encoding` argument is mandatory in practice. Python's default depends on the platform's locale, which on Windows is often `cp1252` and on Linux is usually `utf-8`. A script that omits `encoding` will read fine on the developer's laptop and silently corrupt characters on the production server, or vice versa. Always pass `encoding="utf-8"` for text files unless an upstream system has chosen a different encoding, in which case match it explicitly.

TM1 element names regularly include accented characters, currency symbols, and ideographs, so getting the encoding right matters even for simple dimension element exports. A file written by a TI process with a non Unicode locale may need `encoding="cp1252"` or `encoding="latin-1"` to read; converting it to UTF 8 on the way in is usually worth the extra line.

In [ ]:
src = Path("legacy_export.csv").read_text(encoding="cp1252")
Path("legacy_export_utf8.csv").write_text(src, encoding="utf-8")

Binary mode is the right choice for anything that is not text: a blob uploaded to TM1, a downloaded ZIP, an image. Reading binary into a `str` by accident is one of the more common mistakes (see Topic 14).

## 5. Reading a whole file at once

For a small file, reading the entire contents into memory in one call is the simplest option. `Path.read_text` and `Path.read_bytes` do this and close the file for the caller:

In [ ]:
import json
from pathlib import Path

config: dict[str, object] = json.loads(
    Path("config.json").read_text(encoding="utf-8")
)
config["server"]["address"]   # 'tm1.example.com'

mdx: str = Path("mdx/sales_by_region.mdx").read_text(encoding="utf-8")
cellset = tm1.cubes.cells.execute_mdx(mdx)

`read_text` returns the whole file as a single `str`. `read_bytes` returns it as `bytes`. Both are equivalent to opening the file, calling `.read()`, and closing it, but in one line and with no need for a `with` block.

The threshold for "small enough to read whole" is generous on modern hardware. A 50 MB MDX export string is perfectly reasonable to hold in memory; a 5 GB cube dump is not. The rule is not the absolute size but whether the script needs to scan top to bottom or jump around. If the script only needs the first ten lines, do not read the whole file.

For files written by a TM1 TI process, watch for a UTF 8 byte order mark (`﻿`) at the start of the string. It will throw off a header parse if not handled. Either strip it explicitly or open with `encoding="utf-8-sig"`, which consumes the BOM during decoding.

In [ ]:
text = Path("ti_export.csv").read_text(encoding="utf-8-sig")

## 6. Reading a file line by line

For a file that does not fit comfortably in memory, or where each line is processed independently, iterate the file object directly. Each iteration yields one line, including its trailing newline:

In [ ]:
from pathlib import Path

export_path = Path("exports/sales_plan_2026Q1.csv")
total: float = 0.0

with export_path.open("r", encoding="utf-8") as f:
    header = f.readline()
    for line in f:
        cells = line.rstrip("\n").split(",")
        total += float(cells[-1])

print(total)

The file object is its own iterator. It reads in buffered chunks under the hood and yields one line per loop iteration. Memory use stays flat regardless of file size, which matters for cube exports that can run to hundreds of megabytes.

`line.rstrip("\n")` strips the newline; `line.strip()` would also strip leading whitespace, which is usually wrong for CSV. The CSV section below covers the proper parser; manual `split(",")` is only correct when the data is known to contain no embedded commas or quotes.

A common variant is `readlines()`, which returns a list of all lines at once. It defeats the streaming model and should be avoided for files of any real size; use the loop instead.

## 7. Writing and overwriting

`open(path, "w", ...)` opens a file for writing. If the file already exists, its contents are discarded the moment the file is opened, before any write call. If it does not exist, it is created. `Path.write_text` and `Path.write_bytes` are the one shot equivalents:

In [ ]:
import json
from pathlib import Path

run_record = {
    "run_id": "2026-05-05T08:00:00Z",
    "cube": "Sales Plan",
    "rows_written": 12480,
    "status": "ok",
}

Path("logs/last_run.json").write_text(
    json.dumps(run_record, indent=2),
    encoding="utf-8",
)

The destructive nature of `"w"` is the most important fact about it. A script that opens `audit.log` in `"w"` mode at the top, intending to log to it as the run progresses, will erase every prior run's record before writing its own. The mode for that case is `"a"` (Topic 8).

If the script must not overwrite an existing file, use `"x"`. It opens the file for writing only if creation succeeds, and raises `FileExistsError` otherwise. This is the safe mode for export filenames that include a timestamp or a run id and should never collide:

In [ ]:
export_path = Path(f"exports/sales_plan_{run_id}.csv")
with export_path.open("x", encoding="utf-8") as f:
    f.write("Period,Region,Product,Account,Amount\n")
    for row in rows:
        f.write(",".join(row) + "\n")

For files that are part of an upload (a blob the script feeds to `tm1.processes.execute_with_return` or similar), write to a temporary name first and rename into place once the write is complete. Topic 11 covers that pattern.

## 8. Appending to a log file

An audit log captures what happened on each run: when the job started, which cubes it touched, how many cells it wrote, whether it succeeded. The file grows over time and must never be truncated. Mode `"a"` opens the file for writing without erasing it, and every write goes to the end:

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

log_path = Path("logs/audit.log")

def log_run(record: dict[str, object]) -> None:
    record["timestamp"] = datetime.now(timezone.utc).isoformat()
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

log_run({"run_id": "2026-05-05T08:00:00Z", "status": "ok", "rows": 12480})
log_run({"run_id": "2026-05-05T20:00:00Z", "status": "failed", "error": "view missing"})

Each call opens the file, appends, and closes it. This is more I/O than holding the file open for the life of the script, but the trade off is durability: if the script crashes, every record up to the crash is on disk and flushed. Holding a single handle open and writing to it is faster but loses the last buffered writes if the process is killed.

The format above is JSON Lines: one JSON object per line, separated by newlines. It is the standard format for append friendly logs because each line is a self contained record, and tools like `jq` and `pandas.read_json(lines=True)` consume it directly. Plain text logs are fine too; the choice is JSON Lines when downstream tools will parse the entries, and plain text when a human is the only reader.

## 9. CSV files

CSV is the format TM1 exports and consumes most often. Manual `split(",")` parsing breaks the moment an element name contains a comma, a quote, or a newline. Python's `csv` module handles all of these correctly and is the right tool for any CSV that touches a TM1 cube:

In [ ]:
import csv
from pathlib import Path

export_path = Path("exports/sales_plan_2026Q1.csv")

with export_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # row: {'Period': '2026-Q1', 'Region': 'Europe',
        #       'Product': 'Bicycles', 'Account': 'Revenue',
        #       'Amount': '125000.00'}
        amount = float(row["Amount"])

Two arguments are non obvious. `newline=""` tells `open` not to translate line endings, leaving that to the `csv` module which handles `\r\n` and embedded newlines inside quoted fields correctly. Omitting it produces blank rows on Windows. `encoding="utf-8"` is explicit for the same reason as before.

Writing a CSV uses `csv.writer` or `csv.DictWriter`. For TM1 data, `DictWriter` with named columns is usually the cleanest fit:

In [ ]:
rows: list[dict[str, str]] = [
    {"Period": "2026-Q1", "Region": "Europe", "Product": "Bicycles",
     "Account": "Revenue", "Amount": "125000.00"},
    {"Period": "2026-Q1", "Region": "Europe", "Product": "Bicycles",
     "Account": "Cost", "Amount": "82000.00"},
]

with export_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Period", "Region", "Product", "Account", "Amount"])
    writer.writeheader()
    writer.writerows(rows)

For larger data and any column that needs typing or aggregation, `pandas.read_csv` and `DataFrame.to_csv` are usually the better choice. The `csv` module is the right tool when streaming row by row, when memory matters, or when no other dependency is available.

## 10. Listing a directory and globbing

A scheduled job often processes "every CSV that arrived in the inbox since the last run". `Path.iterdir` lists the immediate children of a directory. `Path.glob` matches a wildcard pattern. `Path.rglob` recurses into subdirectories:

In [ ]:
from pathlib import Path

inbox = Path("/var/lib/tm1py/inbox")

for entry in inbox.iterdir():
    print(entry.name, entry.is_file(), entry.stat().st_size)

for csv_file in sorted(inbox.glob("sales_plan_*.csv")):
    process(csv_file)

for mdx_file in Path("mdx").rglob("*.mdx"):
    print(mdx_file)

`glob` returns paths in the order the operating system reports them, which on Linux is the order they were created in the directory and on other systems is undefined. Sort the result whenever the order matters; relying on directory order to imply chronology is a recurring source of bugs in batch jobs.

`Path.stat()` returns a `os.stat_result` with size, modification time, and permissions. `Path.stat().st_mtime` is a Unix timestamp; convert it with `datetime.fromtimestamp` if a readable value is needed. To pick up only files modified since the last run, compare `st_mtime` against a stored watermark.

`Path.glob` patterns use shell wildcards: `*` matches any sequence except `/`, `?` matches one character, `**` with `rglob` matches any number of directories. They do not understand regex. For anything more sophisticated, list with `iterdir` and filter in Python.

## 11. Atomic writes

A tm1py script that writes an export is sometimes interrupted: the network drops, the user kills the process, the disk fills. If the script writes directly to `sales_plan_2026Q1.csv`, a downstream consumer may pick up a half written file and treat it as complete. The fix is to write to a temporary path in the same directory and rename it into place after the write closes successfully:

In [ ]:
import csv
from pathlib import Path

export_path = Path("exports/sales_plan_2026Q1.csv")
tmp_path = export_path.with_suffix(export_path.suffix + ".tmp")

with tmp_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Period", "Region", "Product", "Account", "Amount"])
    writer.writeheader()
    writer.writerows(rows)

tmp_path.replace(export_path)

`Path.replace` is an atomic rename on the same filesystem: either the destination has the old contents or the new contents; it is never half written from the consumer's perspective. The temporary file lives in the same directory as the destination, because `replace` across filesystems falls back to a copy and is no longer atomic.

This pattern is the right default for any export that another process might be tailing, and for any file where partial contents would be worse than no file at all. It does not help if the disk fills mid write; the temporary file stays behind and should be cleaned up on retry.

## 12. The working directory and resolving relative paths

A relative path like `Path("config.json")` is resolved against the process's current working directory, not against the directory where the `.py` file lives. A script that works when run from its own directory will fail when launched from a scheduler that sets the working directory elsewhere.

In [ ]:
from pathlib import Path

# Where the script file is on disk:
script_dir = Path(__file__).resolve().parent

# Where the process was launched from:
import os
cwd = Path(os.getcwd())

config_path = script_dir / "config.json"   # robust against cwd changes
log_path = script_dir / "logs" / "audit.log"

`__file__` is the path of the module being executed. `.resolve().parent` gives the absolute directory of the script, regardless of how it was launched. Anchoring all file paths to `script_dir` is the simplest way to make a tm1py job portable across the developer's laptop, a CI runner, and a scheduled task on a Windows server, all of which will set different working directories.

A different style is to take all paths from the config file. The script then needs only one well known path, `config.json`, and reads everything else from there. Either approach works; what matters is being deliberate about it. A script that mixes `Path("config.json")` and `Path("/srv/tm1py/exports")` and `Path(__file__).parent / "mdx"` will eventually produce a "file not found" error that depends on who launched it.

## 13. Real world design principles

Patterns that consistently pay off in tm1py file handling:

**Use `pathlib`, not strings.** A `Path` is platform aware, immutable, and has the methods you usually want already. Mixing `os.path.join` and string concatenation across a script invites separator and trailing slash bugs.

**Always pass `encoding`.** Default encoding is platform dependent and silently varies between dev and prod. UTF 8 is the right default for new files; existing files inherit whatever encoding their source system used and must be opened with that encoding explicitly.

**Open with `with`.** Every `open` should be inside a `with` block. The exception is short scripts where `Path.read_text` or `Path.write_text` already encapsulate the open and close.

**Anchor paths to the script or to the config.** Never trust the working directory. Pick `Path(__file__).resolve().parent` or a config provided base path and join from there.

**Append for logs, replace for exports.** An audit log is opened in `"a"` mode for the lifetime of the project; an export is written to a temporary file and renamed into place. Mixing the two patterns is how logs get truncated and how consumers read half written exports.

**Stream when the file is large, slurp when it is small.** A 1 KB config file should be read whole; a 500 MB cube dump should be iterated line by line. The threshold is whether the script needs to hold the file in memory at all once it has the answer it wants.

**Treat element names as case sensitive.** A CSV field that says `"europe"` will not match a TM1 element named `"Europe"` when written through `tm1.cells.write_values`. The file layer does no normalization; if normalization is needed, do it explicitly before handing values to tm1py.

## 14. Common mistakes

Most file handling bugs in tm1py scripts come from a small set of recurring errors. Each of these is worth a moment's attention.

**Forgetting `encoding`.** Default encoding varies by platform.

In [ ]:
# Wrong
with open("config.json") as f:
    text = f.read()

# Correct
with open("config.json", encoding="utf-8") as f:
    text = f.read()

**Opening for write when meaning to append.** `"w"` truncates the file before the first write happens.

In [ ]:
# Wrong: erases every prior audit record on every run
with open("audit.log", "w", encoding="utf-8") as f:
    f.write(record + "\n")

# Correct
with open("audit.log", "a", encoding="utf-8") as f:
    f.write(record + "\n")

**Opening without `with`.** A bare `open` leaks the handle and on Windows locks the file.

In [ ]:
# Wrong
f = open("config.json", encoding="utf-8")
text = f.read()

# Correct
with open("config.json", encoding="utf-8") as f:
    text = f.read()

**Reading binary into text mode.** A `.zip` or `.xlsx` opened in text mode raises `UnicodeDecodeError` immediately, but a `.csv` written in `cp1252` opened as `utf-8` may decode silently and corrupt accented element names.

In [ ]:
# Wrong
text = Path("forecast_upload.zip").read_text(encoding="utf-8")

# Correct
data = Path("forecast_upload.zip").read_bytes()

**Manual CSV parsing.** `split(",")` breaks on quoted fields and embedded commas, both of which appear in real TM1 element names.

In [ ]:
# Wrong
for line in f:
    cells = line.strip().split(",")

# Correct
reader = csv.DictReader(f)
for row in reader:
    ...

**Relying on the working directory.** A path like `Path("config.json")` works from the script's directory and fails from a scheduler.

In [ ]:
# Wrong
config = json.loads(Path("config.json").read_text(encoding="utf-8"))

# Correct
script_dir = Path(__file__).resolve().parent
config = json.loads((script_dir / "config.json").read_text(encoding="utf-8"))

**Direct overwrite of an export.** A consumer tailing the file may read a half written copy.

In [ ]:
# Wrong
with export_path.open("w", encoding="utf-8", newline="") as f:
    writer.writerows(rows)

# Correct
tmp = export_path.with_suffix(export_path.suffix + ".tmp")
with tmp.open("w", encoding="utf-8", newline="") as f:
    writer.writerows(rows)
tmp.replace(export_path)

**Forgetting `newline=""` with the `csv` module on Windows.** Without it, every other row in the output is blank.

In [ ]:
# Wrong
with export_path.open("w", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)

# Correct
with export_path.open("w", encoding="utf-8", newline="") as f:
    csv.writer(f).writerows(rows)